In [1]:
import pandas as pd
import numpy as np
import requests
import os
import datetime as dt
import re
import ast

pd.set_option('display.max_columns', 999)

In [2]:
#Helper Functions

def convert_to_num(df):
    for col in df.columns:
        try:
            pd.to_numeric(df[col])
        except:
            print(f"Column {col} cannot be converted to numeric")
            continue

def labelDom(row):
    if row['DOM'] < 14:
        return True
    else:
        return False

def binaryDom(df):
    df['binaryDOM'] = df.apply(lambda row: labelDom(row), axis = 1)

def createDOM(df: pd.DataFrame, timestamp = False, convert = False):
    dom = df['soldDate'] - df['listDate']
    df['DOM'] = dom.astype("string")
    df.dropna(subset=['DOM'], inplace = True)
    df['DOM'] = df['DOM'].str.slice_replace(start=2)
    df['DOM'] = df['DOM'].astype("int")

def dfRemoveNA(df):
    df = df.replace('', np.nan)
    if sum(df.isna().sum()) != 0:
        df = df.dropna()
    return df

def dfFillNA(df, column):
    df = df.replace('', np.nan)
    df = df.fillna(value = {column: "No "+column})
    return df

def dfExtractFeatures(df: pd.DataFrame, key: str, columns: list):
    if type(df[key].values[0]) == str:
        serie = df[key].apply(lambda x: ast.literal_eval(str(x)))
        d = pd.DataFrame.from_dict(serie.tolist())
    elif type(df[key].values[0]) == dict:
        serie = pd.Series.to_dict(df[key])
        d = pd.DataFrame.from_dict(serie, orient='index')
    for i in range(len(columns)):
        df[columns[i]] = d[columns[i]]

    return df

def dfPriceDifference(df: pd.DataFrame):
    #should only be for clustering purposes only
    df['PriceDifference'] = df['soldPrice'] - df['listPrice']

def dfDatesRemoveTimeStamp(df: pd.DataFrame):
    df['listDate'] = df['listDate'].str.slice_replace(start = 10, repl="")
    df['soldDate'] = df['soldDate'].str.slice_replace(start = 10, repl="")

def dfConvertToDatetime(df: pd.DataFrame, timestamp = False):
    if type(df['listDate'].values[0]) == str:
        dfDatesRemoveTimeStamp(df)
    df['listDate'] = pd.to_datetime(df['listDate'])
    df['soldDate'] = pd.to_datetime(df['soldDate'])

def removeFeatures(df: pd.DataFrame, columns: list):
    #df = df.drop(['images', 'resource', 'timestamps','permissions','lastStatus','status','boardId','agents'], axis = 1)
    df = df.drop(columns, axis = 1)
    return df

#Data Augmentation

def countLevels(df: pd.DataFrame, column):
    series = df[column].value_counts()
    return series

def augmentCategorical(df, column, percent = 0.04):
    series = countLevels(df, column)
    total = np.sum(series.values)
    percentage = [x/total for x in series.values]
    
    for i in range(len(percentage)):
        if percentage[i] > percent:
            pass
        else:
            df = df.replace(to_replace = {column: series.index[i]}, value = 'Other '+column)

    return df

def augmentType(df, type, columns):
    for col in columns:
        df[col] = df[col].astype(df[col].astype(type))

    return df

def augmentNumericals(df):
    for i in range(len(df.columns)):
        if (type(df[df.columns[i]].values[0]) == str):
            numerical = re.search('^[-+]?[0-9]*\.?[0-9]+$', df[df.columns[i]].values[0])
            if numerical is not None:
                df[df.columns[i]] = df[df.columns[i]].astype(float)
    return df

def removeLowLevels(df, column):
    lst = []
    for i in range(len(df[column].value_counts().index)):
        if df[column].value_counts().values[i] < 11:
            lst.append(df[column].value_counts().index[i])

    for i in range(len(lst)):
        df = df[df[column] != lst[i]]
    return df

def removeOutliers(df):
    percentile90 = df['soldPrice'].quantile(0.90)
    percentile10 = df['soldPrice'].quantile(0.10)

    iqr = percentile90 - percentile10

    upper_limit = percentile90 + 3 * iqr
    lower_limit = percentile10 - 3 * iqr

    # df[df['soldPrice'] > upper_limit]
    # df[df['soldPrice'] < lower_limit]

    df = df[df['soldPrice'] < upper_limit]
    df = df[df['soldPrice'] > lower_limit]

    return df


In [3]:
raw_lease_df = pd.read_csv(os.getcwd()+"\\data\\raw_lease_data.csv")

In [4]:
raw_lease_df.columns

Index(['listDate', 'images', 'address', 'soldDate', 'resource', 'timestamps',
       'type', 'mlsNumber', 'permissions', 'soldPrice', 'details', 'class',
       'map', 'listPrice', 'lastStatus', 'status', 'boardId', 'agents'],
      dtype='object')

In [5]:
pd.unique(raw_lease_df['lastStatus'].values)

array(['Lsd', 'Exp', 'Sus', 'Ter', 'Lc', 'Sld'], dtype=object)

In [6]:
raw_lease_df.head()

,listDate,images,address,soldDate,resource,timestamps,type,mlsNumber,permissions,soldPrice,details,class,map,listPrice,lastStatus,status,boardId,agents
0,2019-04-16T00:00:00.000Z,"['IMG-N4418450_1.jpg', 'IMG-N4418450_2.jpg', '...","{'area': 'York', 'zip': 'L3Y0E1', 'country': N...",2019-05-06T00:00:00.000Z,Property:2381,{'listingUpdated': '2019-05-08T10:56:42.000Z'},Lease,N4418450,"{'displayAddressOnInternet': 'Y', 'displayPubl...",2200.0,"{'numBathrooms': '4', 'numBedrooms': '4', 'sqf...",ResidentialProperty,"{'latitude': '44.05193999999999', 'longitude':...",2100.0,Lsd,U,2,[]
1,2019-05-01T00:00:00.000Z,"['IMG-N4435010_1.jpg', 'IMG-N4435010_2.jpg', '...","{'area': 'York', 'zip': 'L4H1T3', 'country': N...",2019-05-06T00:00:00.000Z,Property:2381,{'listingUpdated': '2019-05-08T15:58:16.000Z'},Lease,N4435010,"{'displayAddressOnInternet': 'Y', 'displayPubl...",1350.0,"{'numBathrooms': '1', 'numBedrooms': '2', 'sqf...",ResidentialProperty,"{'latitude': '43.8168699', 'longitude': '-79.6...",1350.0,Lsd,U,2,[]
2,2019-05-03T00:00:00.000Z,"['IMG-N4437661_1.jpg', 'IMG-N4437661_2.jpg', '...","{'area': 'York', 'zip': 'L4E0T9', 'country': N...",2019-05-06T00:00:00.000Z,Property:2381,{'listingUpdated': '2019-05-07T11:06:02.000Z'},Lease,N4437661,"{'displayAddressOnInternet': 'Y', 'displayPubl...",3000.0,"{'numBathrooms': '4', 'numBedrooms': '5', 'sqf...",ResidentialProperty,"{'latitude': '43.9175319', 'longitude': '-79.4...",2999.0,Lsd,U,2,[]
3,2019-04-03T00:00:00.000Z,"['IMG-N4405561_1.jpg', 'IMG-N4405561_2.jpg', '...","{'area': 'York', 'zip': 'L9N 0T3', 'country': ...",2019-05-06T00:00:00.000Z,Property:2381,{'listingUpdated': '2019-05-08T14:23:19.000Z'},Lease,N4405561,"{'displayAddressOnInternet': 'Y', 'displayPubl...",2400.0,"{'numBathrooms': '4', 'numBedrooms': '4', 'sqf...",ResidentialProperty,"{'latitude': '44.09185', 'longitude': '-79.501...",2400.0,Lsd,U,2,[]
4,2019-04-24T00:00:00.000Z,['IMG-W4426017_1.jpg'],"{'area': 'Peel', 'zip': 'L4X4H2', 'country': N...",2019-05-06T00:00:00.000Z,Property:2381,{'listingUpdated': '2019-05-09T13:23:15.000Z'},Lease,W4426017,"{'displayAddressOnInternet': 'Y', 'displayPubl...",3400.0,"{'numBathrooms': '4', 'numBedrooms': '4', 'sqf...",ResidentialProperty,"{'latitude': '43.6064424', 'longitude': '-79.6...",3400.0,Lsd,U,2,[]


The datatypes are mainly objects, but there are dictionaries and floats/integers that are located inside each objects that may need to be converted. Certain predictors like number of bathrooms can be left as strings, 

In [7]:
raw_lease_df.dtypes

listDate        object
images          object
address         object
soldDate        object
resource        object
timestamps      object
type            object
mlsNumber       object
permissions     object
soldPrice      float64
details         object
class           object
map             object
listPrice      float64
lastStatus      object
status          object
boardId          int64
agents          object
dtype: object

In [8]:
address = ['area', 'city', 'district', 'neighborhood']
details = ['numBathrooms','numBedrooms','sqft','style',]
map_ = ['latitude', 'longitude']
columns = ['listDate','soldDate','address','details','images', 'resource', 'timestamps','permissions','lastStatus','status','boardId','agents','type','mlsNumber','map']

lease_df = dfExtractFeatures(raw_lease_df, key = 'address', columns = address)
lease_df = dfExtractFeatures(lease_df, key = 'details', columns = details)
lease_df = dfExtractFeatures(lease_df, key = 'map', columns = map_)

In [9]:
lease_df.head()

,listDate,images,address,soldDate,resource,timestamps,type,mlsNumber,permissions,soldPrice,details,class,map,listPrice,lastStatus,status,boardId,agents,area,city,district,neighborhood,numBathrooms,numBedrooms,sqft,style,latitude,longitude
0,2019-04-16T00:00:00.000Z,"['IMG-N4418450_1.jpg', 'IMG-N4418450_2.jpg', '...","{'area': 'York', 'zip': 'L3Y0E1', 'country': N...",2019-05-06T00:00:00.000Z,Property:2381,{'listingUpdated': '2019-05-08T10:56:42.000Z'},Lease,N4418450,"{'displayAddressOnInternet': 'Y', 'displayPubl...",2200.0,"{'numBathrooms': '4', 'numBedrooms': '4', 'sqf...",ResidentialProperty,"{'latitude': '44.05193999999999', 'longitude':...",2100.0,Lsd,U,2,[],York,Newmarket,Newmarket,Glenway Estates,4,4,1500-2000,3-Storey,44.05193999999999,-79.49083999999999
1,2019-05-01T00:00:00.000Z,"['IMG-N4435010_1.jpg', 'IMG-N4435010_2.jpg', '...","{'area': 'York', 'zip': 'L4H1T3', 'country': N...",2019-05-06T00:00:00.000Z,Property:2381,{'listingUpdated': '2019-05-08T15:58:16.000Z'},Lease,N4435010,"{'displayAddressOnInternet': 'Y', 'displayPubl...",1350.0,"{'numBathrooms': '1', 'numBedrooms': '2', 'sqf...",ResidentialProperty,"{'latitude': '43.8168699', 'longitude': '-79.6...",1350.0,Lsd,U,2,[],York,Vaughan,Vaughan,Sonoma Heights,1,2,,2-Storey,43.8168699,-79.618265
2,2019-05-03T00:00:00.000Z,"['IMG-N4437661_1.jpg', 'IMG-N4437661_2.jpg', '...","{'area': 'York', 'zip': 'L4E0T9', 'country': N...",2019-05-06T00:00:00.000Z,Property:2381,{'listingUpdated': '2019-05-07T11:06:02.000Z'},Lease,N4437661,"{'displayAddressOnInternet': 'Y', 'displayPubl...",3000.0,"{'numBathrooms': '4', 'numBedrooms': '5', 'sqf...",ResidentialProperty,"{'latitude': '43.9175319', 'longitude': '-79.4...",2999.0,Lsd,U,2,[],York,Richmond Hill,Richmond Hill,Jefferson,4,5,3000-3500,2-Storey,43.9175319,-79.43661039999999
3,2019-04-03T00:00:00.000Z,"['IMG-N4405561_1.jpg', 'IMG-N4405561_2.jpg', '...","{'area': 'York', 'zip': 'L9N 0T3', 'country': ...",2019-05-06T00:00:00.000Z,Property:2381,{'listingUpdated': '2019-05-08T14:23:19.000Z'},Lease,N4405561,"{'displayAddressOnInternet': 'Y', 'displayPubl...",2400.0,"{'numBathrooms': '4', 'numBedrooms': '4', 'sqf...",ResidentialProperty,"{'latitude': '44.09185', 'longitude': '-79.501...",2400.0,Lsd,U,2,[],York,East Gwillimbury,East Gwillimbury,Holland Landing,4,4,3000-3500,2-Storey,44.09185,-79.50193
4,2019-04-24T00:00:00.000Z,['IMG-W4426017_1.jpg'],"{'area': 'Peel', 'zip': 'L4X4H2', 'country': N...",2019-05-06T00:00:00.000Z,Property:2381,{'listingUpdated': '2019-05-09T13:23:15.000Z'},Lease,W4426017,"{'displayAddressOnInternet': 'Y', 'displayPubl...",3400.0,"{'numBathrooms': '4', 'numBedrooms': '4', 'sqf...",ResidentialProperty,"{'latitude': '43.6064424', 'longitude': '-79.6...",3400.0,Lsd,U,2,[],Peel,Mississauga,Mississauga,Hurontario,4,4,,2-Storey,43.6064424,-79.6422499


Feature Engineering

In [10]:
lease_df.replace({"": np.nan}, inplace = True)
lease_df.replace({float("nan"): np.nan}, inplace = True)

In [11]:
dfConvertToDatetime(lease_df)

In [12]:
createDOM(lease_df)

In [13]:
lease_df['avg_sqft'] = [(int(value.split("-")[0])+int(value.split("-")[1]))/2 if not isinstance(value, float) and '-' in value else np.nan for value in lease_df['sqft']]

In [14]:
lease_df['avg_sqft']

0         1750.0
1            NaN
2         3250.0
3         3250.0
4            NaN
           ...  
320766     949.5
320767       NaN
320768     549.5
320769     649.5
320770     749.5
Name: avg_sqft, Length: 320770, dtype: float64

In [15]:
lease_df['ppsqft'] = lease_df['listPrice']/lease_df['avg_sqft']

In [16]:
lease_df['ppsqft']

0         1.200000
1              NaN
2         0.922769
3         0.738462
4              NaN
            ...   
320766    4.739336
320767         NaN
320768    4.549591
320769    4.926867
320770    3.202135
Name: ppsqft, Length: 320770, dtype: float64

In [17]:
lease_df['bathtobed_ratio'] = lease_df['numBathrooms'].astype("Int64")/lease_df['numBedrooms'].astype("Int64")

In [18]:
lease_df['latitude'] = lease_df['latitude'].astype("float")
lease_df['longitude'] = lease_df['longitude'].astype("float")

In [19]:
lease_df.tail()

,listDate,images,address,soldDate,resource,timestamps,type,mlsNumber,permissions,soldPrice,details,class,map,listPrice,lastStatus,status,boardId,agents,area,city,district,neighborhood,numBathrooms,numBedrooms,sqft,style,latitude,longitude,DOM,avg_sqft,ppsqft,bathtobed_ratio
320766,2023-03-14,"['IMG-W5965967_1.jpg', 'IMG-W5965967_2.jpg', '...","{'area': 'Toronto', 'zip': 'M6S 1A1', 'country...",2023-04-18,Property:2381,{'listingUpdated': '2023-04-19T21:21:01.000Z'},Lease,W5965967,"{'displayAddressOnInternet': 'Y', 'displayPubl...",4500.0,"{'numBathrooms': '3', 'numBedrooms': '3', 'sqf...",CondoProperty,"{'latitude': '43.6357626', 'longitude': '-79.4...",4500.0,Lsd,U,2,[],Toronto,Toronto,Toronto W01,High Park-Swansea,3,3,900-999,Apartment,43.635763,-79.467490,35,949.5,4.739336,1.0
320767,2023-03-23,"['IMG-W5985741_1.jpg', 'IMG-W5985741_2.jpg', '...","{'area': 'Toronto', 'zip': 'M6P2G3', 'country'...",2023-04-18,Property:2381,{'listingUpdated': '2023-04-19T17:01:48.000Z'},Lease,W5985741,"{'displayAddressOnInternet': 'Y', 'displayPubl...",3700.0,"{'numBathrooms': '1', 'numBedrooms': '2', 'sqf...",ResidentialProperty,"{'latitude': '43.6607634', 'longitude': '-79.4...",3550.0,Lsd,U,2,[],Toronto,Toronto,Toronto W02,High Park North,1,2,NaN,2-Storey,43.660763,-79.460102,26,NaN,NaN,0.5
320768,2023-03-20,"['IMG-C5975287_1.jpg', 'IMG-C5975287_2.jpg', '...","{'area': 'Toronto', 'zip': 'M2J1M5', 'country'...",2023-04-18,Property:2381,{'listingUpdated': '2023-04-19T22:13:37.000Z'},Lease,C5975287,"{'displayAddressOnInternet': 'Y', 'displayPubl...",2500.0,"{'numBathrooms': '1', 'numBedrooms': '1', 'sqf...",CondoProperty,"{'latitude': '43.7716467', 'longitude': '-79.3...",2500.0,Lsd,U,2,[],Toronto,Toronto,Toronto C15,Henry Farm,1,1,500-599,Apartment,43.771647,-79.344663,29,549.5,4.549591,1.0
320769,2023-04-13,"['IMG-C6024761_1.jpg', 'IMG-C6024761_2.jpg', '...","{'area': 'Toronto', 'zip': 'M5H0B1', 'country'...",2023-04-18,Property:2381,{'listingUpdated': '2023-04-19T22:49:33.000Z'},Lease,C6024761,"{'displayAddressOnInternet': 'Y', 'displayPubl...",3500.0,"{'numBathrooms': '1', 'numBedrooms': '1', 'sqf...",CondoProperty,"{'latitude': '43.6505301', 'longitude': '-79.3...",3200.0,Lsd,U,2,[],Toronto,Toronto,Toronto C01,Bay Street Corridor,1,1,600-699,Apartment,43.650530,-79.382148,5,649.5,4.926867,1.0
320770,2023-04-04,"['IMG-S6014669_1.jpg', 'IMG-S6014669_2.jpg', '...","{'area': 'Simcoe', 'zip': 'L4M 6H9', 'country'...",2023-04-18,Property:2381,{'listingUpdated': '2023-04-19T07:27:53.000Z'},Lease,S6014669,"{'displayAddressOnInternet': 'Y', 'displayPubl...",2400.0,"{'numBathrooms': '1', 'numBedrooms': '1', 'sqf...",CondoProperty,"{'latitude': '44.3896491', 'longitude': '-79.6...",2400.0,Lsd,U,2,[],Simcoe,Barrie,Barrie,North Shore,1,1,700-799,Apartment,44.389649,-79.684525,14,749.5,3.202135,1.0


In [20]:
avg_price_by_area = lease_df.groupby('area').agg('mean')['listPrice']
avg_price_by_city = lease_df.groupby('city').agg('mean')['listPrice']

In [21]:
lease_df_agg = lease_df.join(avg_price_by_area, on = 'area', rsuffix='_by_area')
lease_df_agg = lease_df_agg.join(avg_price_by_city, on = 'city', rsuffix='_by_city')

In [22]:
columns_to_drop = ['listDate', 'images', 'address', 'soldDate', 'resource' ,'timestamps', 'type', 'mlsNumber' ,'permissions' ,'details', 'map' ,'lastStatus', 'status', 'boardId' ,'agents']

In [23]:
lease_df_agg = lease_df_agg.drop(columns = columns_to_drop, axis = 1)

In [24]:
lease_df_agg.head()

,soldPrice,class,listPrice,area,city,district,neighborhood,numBathrooms,numBedrooms,sqft,style,latitude,longitude,DOM,avg_sqft,ppsqft,bathtobed_ratio,listPrice_by_area,listPrice_by_city
0,2200.0,ResidentialProperty,2100.0,York,Newmarket,Newmarket,Glenway Estates,4,4,1500-2000,3-Storey,44.051940,-79.490840,20,1750.0,1.200000,1.0,2602.068714,2404.587468
1,1350.0,ResidentialProperty,1350.0,York,Vaughan,Vaughan,Sonoma Heights,1,2,NaN,2-Storey,43.816870,-79.618265,5,NaN,NaN,0.5,2602.068714,2600.925518
2,3000.0,ResidentialProperty,2999.0,York,Richmond Hill,Richmond Hill,Jefferson,4,5,3000-3500,2-Storey,43.917532,-79.436610,3,3250.0,0.922769,0.8,2602.068714,2679.261341
3,2400.0,ResidentialProperty,2400.0,York,East Gwillimbury,East Gwillimbury,Holland Landing,4,4,3000-3500,2-Storey,44.091850,-79.501930,33,3250.0,0.738462,1.0,2602.068714,2784.416058
4,3400.0,ResidentialProperty,3400.0,Peel,Mississauga,Mississauga,Hurontario,4,4,NaN,2-Storey,43.606442,-79.642250,12,NaN,NaN,1.0,2549.302092,2565.449606


Encoding to categorical variables

In [25]:
lease_df_agg_cats = pd.get_dummies(lease_df_agg)

In [26]:
lease_df_agg_cats.head()

,soldPrice,listPrice,latitude,longitude,DOM,avg_sqft,ppsqft,bathtobed_ratio,listPrice_by_area,listPrice_by_city,class_CondoProperty,class_ResidentialProperty,area_Algoma,area_Brant,area_Brantford,area_Bruce,area_Canada,area_Chatham-Kent,area_Cochrane,area_Dufferin,area_Durham,area_Elgin,area_Essex,area_Frontenac,area_Greater Sudbur,area_Grey County,area_Haldimand,area_Halton,area_Hamilton,area_Hastings,area_Huron,area_Kawartha Lakes,area_Lambton,area_Lanark,area_Leeds & Grenvi,area_Lennox & Addin,area_Manitoulin,area_Middlesex,area_Muskoka,area_Niagara,area_Nipissing,area_Norfolk,area_Northumberland,area_Ottawa,area_Oxford,area_Parry Sound,area_Peel,area_Perth,area_Peterborough,area_Prince Edward,area_Renfrew,area_Simcoe,"area_Stormont, Dund",area_Sudbury,area_Toronto,area_Waterloo,area_Wellington,area_York,city_Adjala-Tosorontio,city_Ajax,city_Alnwick/Haldimand,city_Amaranth,city_Amherstburg,city_Arnprior,city_Ashfield-Colborne-Wawanosh,city_Asphodel-Norwood,city_Aurora,city_Barrie,city_Belleville,city_Blue Mountains,city_Bluewater,city_Bracebridge,city_Bradford West Gwillimbury,city_Brampton,city_Brant,city_Brantford,city_Brighton,city_Brock,city_Brockton,city_Brockville,city_Burlington,city_Caledon,city_Cambridge,city_Carleton Place,city_Cavan Monaghan,city_Central Frontenac,city_Centre Wellington,city_Chatham-Kent,city_Clarington,city_Clearview,city_Cobourg,city_Collingwood,city_Cornwall,city_Cramahe,city_Douro-Dummer,city_Dutton/Dunwich,city_East Garafraxa,city_East Gwillimbury,city_East Luther Grand Valley,city_East Zorra-Tavistock,city_Erin,city_Essa,city_Fort Erie,city_Galway-Cavendish and Harvey,city_Georgian Bay,city_Georgian Bluffs,city_Georgina,city_Goderich,city_Gravenhurst,city_Greater Napanee,city_Greater Sudbury,city_Grey Highlands,city_Grimsby,city_Guelph,city_Guelph/Eramosa,city_Haldimand,city_Halton Hills,city_Hamilton,city_Hamilton Township,city_Hanover,city_Havelock-Belmont-Methuen,city_Huntsville,city_Ingersoll,city_Innisfil,city_Kawartha Lakes,city_Kincardine,city_King,city_Kingston,city_Kitchener,city_LaSalle,city_Lakeshore,city_Lambton Shores,city_Lincoln,city_London,city_Loyalist,city_Lucan Biddulph,city_Madoc,city_Mapleton,city_Markham,city_Marmora and Lake,city_Meaford,city_Melancthon,city_Middlesex Centre,city_Midland,city_Milton,city_Minto,city_Mississauga,city_Mississippi Mills,city_Mono,city_Mulmur,city_Muskoka Lakes,city_New Tecumseth,city_Newmarket,city_Niagara Falls,city_Niagara-on-the-Lake,city_Norfolk,city_North Dumfries,city_North Grenville,city_North Middlesex,city_North Perth,city_Northeastern Manitoulin and,city_Norwich,city_Oakville,city_Orangeville,city_Orillia,city_Oro-Medonte,city_Oshawa,city_Otonabee-South Monaghan,city_Ottawa,city_Out of Area,city_Owen Sound,city_Pelham,city_Penetanguishene,city_Peterborough,city_Petrolia,city_Pickering,city_Plympton-Wyoming,city_Point Edward,city_Port Colborne,city_Port Hope,city_Prince Edward County,city_Puslinch,city_Quinte West,city_Ramara,city_Richmond Hill,city_Rideau Lakes,city_Sarnia,city_Saugeen Shores,city_Sault Ste Marie,city_Scugog,city_Severn,city_Shelburne,city_Smith-Ennismore-Lakefield,city_South Huron,city_South-West Oxford,city_Southgate,city_Southwold,city_Springwater,city_St. Catharines,city_St. Clair,city_St. Marys,city_St. Thomas,city_Stirling-Rawdon,city_Stratford,city_Strathroy-Caradoc,city_Sudbury Remote Area,city_Tay,city_Thorold,city_Tillsonburg,city_Timmins,city_Tiny,city_Toronto,city_Trent Hills,city_Tweed,city_Uxbridge,city_Vaughan,city_Wainfleet,city_Wasaga Beach,city_Waterloo,city_Welland,city_Wellesley,city_Wellington North,city_West Grey,city_West Lincoln,city_West Nipissing,city_Westport,city_Whitby,city_Whitchurch-Stouffville,city_Whitestone,city_Wilmot,city_Windsor,city_Woodstock,city_Woolwich,city_Zorra,district_Adjala-Tosorontio,district_Ajax,district_Alnwick/Haldimand,district_Amaranth,district_Amherstburg,district_Arnprior,district_Ashfield-Colborne-Wawanosh,district_Asphodel-Norwood,district_Aurora,district

In [27]:
lease_df_agg_cats.to_csv(os.getcwd()+"\\data\\data.csv", index = False)